In [2]:
import os

os.chdir(os.getenv("HOME_DIR"))

import random

import onnxruntime as ort
import torch
import yaml
from dotenv import load_dotenv
from ultralytics import YOLO

from src.utils.metrics import fps

load_dotenv()

True

In [3]:
handle_model_yaml = "yolo8_baseline.yaml"
dataset_yaml = "bdd100k.yaml"
yaml_path = os.path.join(
    os.getenv("HOME_DIR"),
    "config",
    "models",
    handle_model_yaml,  # handle_model.yaml
)
with open(yaml_path, "r") as file:
    args = yaml.safe_load(file)

In [4]:
DATA_DIR = os.path.join(
    os.getenv("HOME_DIR"), "config", "datasets", dataset_yaml
)  # Default dataset_name.yaml or personal_dataset_name.yaml
IMG_SIZE = int(os.getenv("HEIGHT")), int(os.getenv("WIDTH"))
PROJECT_DIR = os.path.join(
    os.getenv("HOME_DIR"), "results", "models", args["project_results_name"]
)
TESTING_IMG_DIR = os.path.join(
    os.getenv("HOME_DIR"), "data", "processed", "images", "test"
)  # Testing images directory

In [5]:
onnx_path = os.path.join(PROJECT_DIR, "optimized", "best_optimized.onnx")
best_model_path = os.path.join(
    PROJECT_DIR,
    "optimized",
    "best_optimized.pt",
)
if not os.path.exists(best_model_path) and not os.path.exists(onnx_path):
    onnx_path = os.path.join(PROJECT_DIR, "train", "weights", "best.onnx")
    best_model_path = os.path.join(
        PROJECT_DIR,
        "train",
        "weights",
        "best.pt",
    )

In [6]:
model_size = os.path.getsize(onnx_path) / (1024 * 1024)  # Size in MB
print(f"Model size: {model_size:.2f} MB")

Model size: 11.62 MB


In [7]:
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
image_path = os.path.join(
    TESTING_IMG_DIR,
    os.listdir(TESTING_IMG_DIR)[random.randint(0, len(os.listdir(TESTING_IMG_DIR)))],
)
result = fps(sess, image_path)
print(f"FPS on CPU (edge simulation): {result['fps']}")

FPS on CPU (edge simulation): 84.01741602472987


In [8]:
best_model = YOLO(best_model_path, task="detect", verbose=True).to(torch.device("cpu"))
real_input_torch = torch.from_numpy(
    result["real_input"].transpose((0, 2, 3, 1))
).permute(0, 3, 1, 2)

with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU]) as prof:
    best_model(real_input_torch)
print(prof.key_averages().table(sort_by="self_cpu_time_total"))


0: 480x480 23 cars, 32.1ms
Speed: 0.1ms preprocess, 32.1ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 480)
-------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                       Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                   aten::mkldnn_convolution        42.91%      20.625ms        43.67%      20.988ms     349.804us            60  
                             aten::uniform_        20.56%       9.883ms        20.56%       9.883ms      86.691us           114  
                                   aten::mm         9.33%       4.487ms         9.37%       4.502ms      39.492us           114  
                                aten::copy_         5.94%       2.855ms         5.94%       2.8